# Lesson 02 - Microsoft Agent Framework അനുഭവപരിചയം

**Microsoft Agent Framework (MAF)** അനേകം AI ഏജന്റുകള്‍ നിര്‍മ്മിക്കുന്നതിന് ഒരു ഏകോപിത ഫ്രെയിംവര്‍ക്ക് ആണ്. ഇതില്‍ നാല് പ്രധാന ഘടകങ്ങളുള്ള ശുചിത്വമുള്ള, ഘടകപരമായ ആസ്ഥാനരൂപരേഖ നല്‍കുന്നു:

- **Client** – AI മോഡല്‍ എന്റെപോയിന്റുമായി ബന്ധപ്പെടുകയും ആശയവിനിമയം കൈകാര്യം ചെയ്യുകയും ചെയ്യുന്നു
- **Agent** – ഉപാധികള്‍ നല്‍കി ഒരു ക്ലയന്റിനെ പൊതിഞ്ഞിരിക്കുന്നു, ഉപകരണ നിര്‍വചനങ്ങളും ഉള്‍പ്പെടുന്നു
- **Tools** – മോഡല്‍ വിളിക്കാവുന്ന കസ്റ്റം ഫംഗ്ഷനുകളുമായി ഏജന്റിന്റെ കഴിവുകള്‍ വിപുലീകരിക്കുന്നു
- **Session** – ബഹു-തുറവുള്ള സംവാദങ്ങളുടെ ചരിത്രം നിലനിര്‍ത്തുന്നു

ഈ പാഠത്തില്‍, നാം ഈ ആശയങ്ങള്‍ ഉപയോഗിച്ച് <strong>യാത്ര ബുക്കിങ് ഏജന്റ്</strong> നിര്‍മ്മിക്കുകയാണു, ഇത് ഗമ്യസ്ഥലത്തിന്റെ ലഭ്യത പരിശോധിക്കുന്നു.


## സെറ്റപ്പ്


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ഏജന്റ് ഫ്രെയിംവർക്ക് ആർക്കിടെക്ചർ മനസിലാക്കൽ

Microsoft Agent Framework ഒരു ലെയർഡ് ആർക്കിടെക്ചർ പിന്തുടരുന്നു:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ക്ലയന്റ്** – ഒരു `AzureAIProjectAgentProvider` Azure OpenAI ഡിപ്പ്ലോയ്‌മെന്റുമായി ബന്ധിപ്പിക്കുന്നു. ഇത് പ്രാമാണീകരണം, അഭ്യർത്ഥന ഫോർമാറ്റ് ചെയ്യൽ, പ്രതികരണ പാഴ്സിംഗ് എന്നിവ കൈകാര്യം ചെയ്യുന്നു.
2. **ഏജന്റ്** – ക്ലയന്റിൽ നിന്നും `provider.create_agent()` വഴിയുള്ള രൂപകൽപ്പന ചെയ്‌ത, ഏജന്റ് മോഡൽ ആക്സസ്, നിർദേശംകൾ (സിസ്റ്റം പ്രോംപ്റ്റ്), ടൂൾസ് എന്നിവ സംയോജിപ്പിക്കുന്നു.
3. **ടൂൾസ്** – `@tool` കൊണ്ട് അലങ്കൃത Python ഫംഗ്ഷനുകൾ, ഏജന്റ് പ്രവർത്തനങ്ങൾ നടത്താൻ അല്ലെങ്കിൽ ഡാറ്റ റീറ്റ്രീവ് ചെയ്യാൻ ഇവ വിളിക്കാവുന്നതാണ്.
4. **സെഷൻ** – ഒരു `AgentSession` ഒബ്ജക്റ്റ് (എജന്റ് വഴി `agent.create_session()` ഉണ്ടാകുന്നത്), സംഭാഷണ ചരിത്രം സൂക്ഷിക്കുന്നു, ഏജന്റ് മുൻപ് ഉള്ള കോൺടക്‌സ്‌റ്റ് ഓർമ്മിക്കുന്ന മൾട്ടി-ടേൺ ഡയലോഗ് സാധ്യമാക്കുന്നു.

ഓരോ ഘട്ടവും യഥാക്രമം നിർമ്മിക്കാം.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ഡെക്കറേറ്ററുമായി ടൂളുകൾ ചേർക്കുന്നു

ടൂളുകൾ എജൻറുകൾക്ക് ടെക്സ്റ്റ് ജനരേറ്റ് ചെയ്യുന്നതിൽപ്പുറം പ്രവർത്തനങ്ങൾ സ്വീകരിക്കാൻ അനുവദിക്കുന്നു. `@tool` ഡെക്കറേറ്റർ സാധാരണമായ പൈറ്റൺ ഫംഗ്ഷനെ എജൻറ് വിളിക്കാൻ കഴിയുന്ന ഒന്നായി മാറുന്നു.

പ്രധാന കാമ്പുകൾ:
- മാദ്ധ്യമം ഓരോ പാരാമീറ്ററും മനസ്സിലാക്കുന്ന വിധത്തിൽ `Annotated[type, "description"]` ഉപയോഗിക്കുക.
- ഡോക്‌സ്റ്റ്രിംഗ് ടൂൾ വിവരണമായി മാദ്ധ്യമം കാണുന്നു.
- `approval_mode="never_require"` എന്നതിന്റെ അർഥം ടൂൾ ഉപയോക്തൃ സ്ഥിരീകരണം ഇല്ലാതെയായി സ്വയം പ്രവർത്തിക്കുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ടൂളുകളോടുകൂടിയ ഒരു ഏജന്റ് സൃഷ്ടിക്കുന്നത്

ഇപ്പോൾ നാം ക്ലയന്റ്, നിർദേശങ്ങൾ, ടൂളുകൾ എന്നിവയെ ഒരു ഏജന്റിൽ സംയോജിപ്പിക്കുന്നു. `instructions` സിസ്റ്റം പ്രോംപ്റ്റിന്റെ പകരമാണ് — അവ ഏജന്റിന്റെ വ്യക്തിത്വവും പെരുമാറ്റവും നിർവ്വചിക്കുന്നു.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## സഷനുകളുമായി മൾട്ടി-ടേൺ സംഭാഷണങ്ങൾ

ഒരു `AgentSession` (`agent.create_session()` വഴി സൃഷ്ടിച്ച) ഒരു സംഭാഷണത്തിലെ എല്ലാ സന്ദേശങ്ങളും പിന്തുടരുന്നു. ഒരേ സഷനെ ഓരോ `agent.run()` കോൾലിലും പാസ്സുചെയ്‌തുകൊണ്ട്, ഏജന്റിന് മുഴുവൻ സംഭാഷണ ചരിത്രവും ലഭ്യമാകുന്നു, കൂടാതെ മുമ്പത്തെയുള്ള സന്ദേശങ്ങളെ അവൻ തിരിച്ചറിയാനാകുന്നു.

നാം `tools=[check_destination_availability]` പാസ്സ് ചെയ്യുന്നു അതിനാൽ ഏജന്റ് എല്ലാ ടേണുകളിലും നമ്മുടെ ലഭ്യത പരിശോധിക്കുന്നതിന് ആവിഷ്‌ക്കാരത്തെ വിളിക്കാനാകുന്നു.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ മൈക്രോസോഫ്‌റ്റ് എജന്റ് ഫ്രെയിംവർക്കിന്റെ നാല് തൂണുകൾ പരിശോധിച്ചു:

| സങ്കൽപ്പം | നിങ്ങൾ പഠിച്ചത് |
|---------|------------------|
| **ക്ലയന്റ്** | `AzureAIProjectAgentProvider` ക്രെഡൻഷ്യൽ അടിസ്ഥാനAuth ഉപയോഗിച്ച് Azure OpenAI-യിൽ കണക്റ്റ് ചെയ്യുന്നു |
| **എജന്റ്** | `provider.create_agent()` ഒരു മോഡൽ കണക്ഷനും നിർദ്ദേശങ്ങളും പേരും അടങ്ങിയ ഘടനയാക്കുന്നു |
| **ടൂളുകൾ** | `@tool` ഡെക്കറേറ്റർ എജന്റ് വിളിക്കാനുള്ള പൈത്തൺ ഫങ്‌ഷനുകൾ പുറത്തുകൊണ്ടുവരുന്നു |
| **സെഷൻ** | `agent.create_session()` നിരവധി ടേണുകളിലായി സംഭാഷണ ചരിത്രം കൈവശം വയ്ക്കുന്നു |

ഈ നിർമ്മാണ ഘടകങ്ങൾ ചേർന്ന് പ്രകൃതസ്വഭാവമുള്ള സംഭാഷണങ്ങൾ നടത്താൻ, ബാഹ്യ ഫങ്‌ഷനുകൾ വിളിക്കാൻ, സംഘർശ്യവും നിലനിർത്താൻ കഴിയുന്ന എജന്റ्स സൃഷ്ടിക്കുന്നു — പിന്നീട് പാഠങ്ങളിൽ കൂടുതൽ ആധുനികമായ എജന്റിക് മാതൃകകളുടെ അടിസ്ഥാനം.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ഡിസ്ക്ലെയിമർ**:  
ഈ ഡോക്യുമെന്റ് AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. നാം ശരിയായി പരിഭാഷ എത്തിക്കാൻ ശ്രമിക്കുമ്പോഴും, സ്വയംകൃതമായ പരിഭാഷകൾക്ക് പിഴവുകൾ അല്ലെങ്കിൽ അസ്ഥിരതകൾ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. അത് മൂലഭാഷയിൽ ഉള്ള ഒറിജിനൽ ഡോക്യുമെന്റ് യഥാർത്ഥ അവലംബസ്രോതസാണ്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മാനവ പരിഭാഷ ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ചതിൽ നിന്നുള്ള ഏതു തെറ്റിദ്ധാരണകൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
